## Imports and Config

In [ ]:
%load_ext autoreload
%autoreload 2

Custom imports from the `config/settings.py` file.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT / "src"))

In [ ]:
import json
import math
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.neighbors import NearestNeighbors

from haystack_integrations.components.retrievers.qdrant import QdrantEmbeddingRetriever

from openai import OpenAI

In [ ]:
from sem_rag.config.settings import VLLM, Qdrant, Evaluation, SEMDataset
from sem_rag.retrieval.bm25 import BM25Retriever
from sem_rag.embedding.embedder import CLIPEmbedder
from sem_rag.retrieval.components import QueryEmbedder, MMRReranker
from sem_rag.image.encoding import encode_image
from sem_rag.storage.qdrant import get_document_store
from sem_rag.image.descriptions import get_crop_path

In [ ]:
EVAL_DEPTH = 50
TOP_K = 5

In [ ]:
client = OpenAI(base_url=VLLM.API_URL, api_key=VLLM.API_KEY)

In [ ]:
qwen_qdrant = get_document_store(Qdrant.COLLECTION_QWEN, Qdrant.EMBEDDING_DIM_QWEN)
clip_qdrant = get_document_store(Qdrant.COLLECTION_CLIP, Qdrant.EMBEDDING_DIM_CLIP)

In [ ]:
bm25_retriever = BM25Retriever(Evaluation.BM25_DIR)
clip_embedder = CLIPEmbedder()
qwen_embedder = QueryEmbedder()

clip_retriever = QdrantEmbeddingRetriever(document_store=clip_qdrant, top_k=EVAL_DEPTH)
qwen_retriever = QdrantEmbeddingRetriever(document_store=qwen_qdrant, top_k=EVAL_DEPTH, return_embedding=True)
mmr = MMRReranker()

```text
broad / exploratory  ◄──────────────────────────────────►  narrow / precise
"show me nanowires"         "rough porous films,          "that SEM image of a nanotube
                              cracked surface"               bridging two electrodes"
many relevant images        some relevant images           exactly one relevant image
```

# 1. Category Retrieval

- Queries: 5 per category (50 in total), describing the category.
- Relevance: an image is relevant if its human label matches the query’s category. No judging needed.
- Metrics: Precision@5/10, nDCG@10 and AP@50, overall and per category.

What it tells you: whether each system understands morphology at the level the dataset was annotated.

Limits to state in the write-up:
- It can’t reward a fine-grained match within a category.
- The labels have some noise.
- Categories are imbalanced (169 Porous_Sponge vs 3,515 MEMS). Per-category results and macro-averaging handle that.

Every image in the SEM dataset has a human-assigned morphology category (`SEMDataset.CATEGORIES`, from Aversa et al.). Each query below targets one category, and a retrieved image is considered **relevant if its category matches the category of the query**. No judge is needed.

Each category has two query styles:
- `named`: uses the common name of the object (e.g. "nanowires"), like a user who knows what they are looking for.
- `descriptive`: only describes what is visible, without naming the category. This tests visual understanding rather than vocabulary matching (BM25 searches Gemma descriptions, which often contain the category name).

Limitations: relevance is coarse (any image of the right category counts) and categories partly overlap visually (e.g. Particles vs Powder).

## Queries

In [ ]:
CATEGORY_QUERIES = [
    # __________ Biological __________
    {"id": "BIO-01", "category": "Biological", "style": "named", "query": "Biological sample."},
    {"id": "BIO-02", "category": "Biological", "style": "named", "query": "SEM image of cells and bacteria on a substrate."},
    {"id": "BIO-03", "category": "Biological", "style": "descriptive", "query": "Clusters of small round bodies connected by thin organic filaments on a flat surface."},
    {"id": "BIO-04", "category": "Biological", "style": "descriptive", "query": "Branching thread-like strands spreading over a surface with small irregular blobs."},
    {"id": "BIO-05", "category": "Biological", "style": "descriptive", "query": "Soft, irregular tissue-like network of tangled strands."},

    # __________ Fibres __________
    {"id": "FIB-01", "category": "Fibres", "style": "named", "query": "Textile fibres."},
    {"id": "FIB-02", "category": "Fibres", "style": "named", "query": "Bundle of fibers seen at high magnification."},
    {"id": "FIB-03", "category": "Fibres", "style": "descriptive", "query": "Long, smooth cylindrical strands lying parallel to each other."},
    {"id": "FIB-04", "category": "Fibres", "style": "descriptive", "query": "Thick twisted cylindrical filaments with smooth surfaces."},
    {"id": "FIB-05", "category": "Fibres", "style": "descriptive", "query": "Large smooth rods crossing each other with some debris on top."},

    # __________ Films_Coated_Surface __________
    {"id": "FCS-01", "category": "Films_Coated_Surface", "style": "named", "query": "Thin film coating on a substrate."},
    {"id": "FCS-02", "category": "Films_Coated_Surface", "style": "named", "query": "Coated surface."},
    {"id": "FCS-03", "category": "Films_Coated_Surface", "style": "descriptive", "query": "Flat, nearly featureless surface with a subtle grainy texture."},
    {"id": "FCS-04", "category": "Films_Coated_Surface", "style": "descriptive", "query": "Continuous layer covering a substrate, with small cracks and defects."},
    {"id": "FCS-05", "category": "Films_Coated_Surface", "style": "descriptive", "query": "Surface made of tightly packed faceted grains."},

    # __________ MEMS_devices_and_electrodes __________
    {"id": "MEMS-01", "category": "MEMS_devices_and_electrodes", "style": "named", "query": "MEMS device."},
    {"id": "MEMS-02", "category": "MEMS_devices_and_electrodes", "style": "named", "query": "Microfabricated electrodes with contact pads."},
    {"id": "MEMS-03", "category": "MEMS_devices_and_electrodes", "style": "descriptive", "query": "Metal contact pads connected by thin conductive lines."},
    {"id": "MEMS-04", "category": "MEMS_devices_and_electrodes", "style": "descriptive", "query": "A single thin wire suspended between two large rectangular pads."},
    {"id": "MEMS-05", "category": "MEMS_devices_and_electrodes", "style": "descriptive", "query": "Chip layout with many lines converging from square pads at the border towards the centre."},

    # __________ Nanowires __________
    {"id": "NW-01", "category": "Nanowires", "style": "named", "query": "Nanowires."},
    {"id": "NW-02", "category": "Nanowires", "style": "named", "query": "Forest of vertically grown nanowires."},
    {"id": "NW-03", "category": "Nanowires", "style": "descriptive", "query": "Dense tangled network of long, thin, curved threads."},
    {"id": "NW-04", "category": "Nanowires", "style": "descriptive", "query": "Thin wires growing out of a surface in random directions."},
    {"id": "NW-05", "category": "Nanowires", "style": "descriptive", "query": "Thin bright wires lying randomly on a dark background."},

    # __________ Particles __________
    {"id": "PAR-01", "category": "Particles", "style": "named", "query": "Nanoparticles dispersed on a surface."},
    {"id": "PAR-02", "category": "Particles", "style": "named", "query": "Spherical particles."},
    {"id": "PAR-03", "category": "Particles", "style": "descriptive", "query": "Many small round dots scattered on a flat background."},
    {"id": "PAR-04", "category": "Particles", "style": "descriptive", "query": "Clusters of small rounded grains aggregated together."},
    {"id": "PAR-05", "category": "Particles", "style": "descriptive", "query": "A few isolated bright spots on a dark, smooth substrate."},

    # __________ Patterned_surface __________
    {"id": "PAT-01", "category": "Patterned_surface", "style": "named", "query": "Lithographically patterned surface."},
    {"id": "PAT-02", "category": "Patterned_surface", "style": "named", "query": "Micropillar array."},
    {"id": "PAT-03", "category": "Patterned_surface", "style": "descriptive", "query": "Regular array of identical features repeated in rows and columns."},
    {"id": "PAT-04", "category": "Patterned_surface", "style": "descriptive", "query": "Parallel grooves or stripes etched into a surface."},
    {"id": "PAT-05", "category": "Patterned_surface", "style": "descriptive", "query": "Periodic holes in a flat layer forming a regular lattice."},

    # __________ Porous_Sponge __________
    {"id": "POR-01", "category": "Porous_Sponge", "style": "named", "query": "Porous material."},
    {"id": "POR-02", "category": "Porous_Sponge", "style": "named", "query": "Sponge-like structure."},
    {"id": "POR-03", "category": "Porous_Sponge", "style": "descriptive", "query": "Surface full of irregular round holes and cavities."},
    {"id": "POR-04", "category": "Porous_Sponge", "style": "descriptive", "query": "Three-dimensional network of thin walls surrounding open voids."},
    {"id": "POR-05", "category": "Porous_Sponge", "style": "descriptive", "query": "Foam-like structure with interconnected pores of different sizes."},

    # __________ Powder __________
    {"id": "POW-01", "category": "Powder", "style": "named", "query": "Powder sample."},
    {"id": "POW-02", "category": "Powder", "style": "named", "query": "Agglomerated powder."},
    {"id": "POW-03", "category": "Powder", "style": "descriptive", "query": "Rough heap of fine grains packed together with no regular arrangement."},
    {"id": "POW-04", "category": "Powder", "style": "descriptive", "query": "Loose granular material made of many small irregular clumps."},
    {"id": "POW-05", "category": "Powder", "style": "descriptive", "query": "Fluffy aggregates of tiny grains filling the whole field of view."},

    # __________ Tips __________
    {"id": "TIP-01", "category": "Tips", "style": "named", "query": "AFM tip."},
    {"id": "TIP-02", "category": "Tips", "style": "named", "query": "Sharp probe tip."},
    {"id": "TIP-03", "category": "Tips", "style": "descriptive", "query": "A single sharp conical needle tapering to a fine point."},
    {"id": "TIP-04", "category": "Tips", "style": "descriptive", "query": "A pointed object with a very fine apex against a uniform background."},
    {"id": "TIP-05", "category": "Tips", "style": "descriptive", "query": "Side view of a cone narrowing to a sharp end."},
]

In [ ]:
print(f"Total queries: {len(CATEGORY_QUERIES)}")

In [ ]:
labels = pd.read_csv(SEMDataset.CATEGORIES, dtype={"TIF_SEMImageID": str})

CATEGORY_OF = dict(zip(labels["TIF_SEMImageID"], labels["mapped_category"]))
CATEGORY_SIZE = labels["mapped_category"].value_counts().to_dict()
N_IMAGES = len(labels)

pd.Series(CATEGORY_SIZE, name="images")

## Retrieval

In [ ]:
def retrieve_all(query: str) -> dict[str, list[str]]:
    """Return the ranked image IDs of every system for a text query."""
    bm25 = [result["image_id"] for result in bm25_retriever.search(query, top_k=EVAL_DEPTH)]

    clip_embedding = clip_embedder.embed_text(query)
    clip = [doc.id for doc in clip_retriever.run(query_embedding=clip_embedding)["documents"]]

    qwen_embedding = qwen_embedder.run(text=query)["embedding"]
    qwen_docs = qwen_retriever.run(query_embedding=qwen_embedding)["documents"]
    system_docs = mmr.run(query_embedding=qwen_embedding, documents=qwen_docs)["documents"]

    return {
        "BM25": bm25,
        "CLIP": clip,
        "Qwen3-VL": [doc.id for doc in qwen_docs],
        "InternRAG": [doc.id for doc in system_docs],
    }

In [ ]:
rankings = {case["id"]: retrieve_all(case["query"]) for case in CATEGORY_QUERIES}

## Metrics

Binary relevance (1 if the image has the query's category, 0 otherwise):
- **P@k**: fraction of the top-k images that are relevant.
- **nDCG@10**: like P@10, but relevant images ranked higher count more.
- **AP@50**: average precision over the top 50, normalised by `min(#relevant, 50)`. It rewards putting relevant images early across the whole list.

The **Random** baseline is the expected score of a random ranking, i.e. the share of the category in the dataset. It matters because categories are imbalanced (MEMS is ~25% of the dataset, Fibres ~1%).

Metrics are `NaN` when a system returns fewer than k images (InternRAG at k > 5).

In [ ]:
def precision_at_k(relevant: list[bool], k: int) -> float:
    if len(relevant) < k:
        return np.nan
    return sum(relevant[:k]) / k


def ndcg_at_k(relevant: list[bool], k: int, n_relevant: int) -> float:
    if len(relevant) < k:
        return np.nan
    dcg = sum(rel / math.log2(i + 2) for i, rel in enumerate(relevant[:k]))
    ideal_dcg = sum(1 / math.log2(i + 2) for i in range(min(n_relevant, k)))
    return dcg / ideal_dcg


def average_precision_at_k(relevant: list[bool], k: int, n_relevant: int) -> float:
    if len(relevant) < k:
        return np.nan
    hits, total = 0, 0.0
    for i, rel in enumerate(relevant[:k], start=1):
        if rel:
            hits += 1
            total += hits / i
    return total / min(n_relevant, k)

In [ ]:
rows = []

for case in CATEGORY_QUERIES:
    n_relevant = CATEGORY_SIZE[case["category"]]
    prior = n_relevant / N_IMAGES

    for system, ranking in rankings[case["id"]].items():
        relevant = [CATEGORY_OF.get(image_id) == case["category"] for image_id in ranking]

        rows.append({
            "id": case["id"],
            "category": case["category"],
            "style": case["style"],
            "system": system,
            "P@5": precision_at_k(relevant, 5),
            "P@10": precision_at_k(relevant, 10),
            "nDCG@10": ndcg_at_k(relevant, 10, n_relevant),
            "AP@50": average_precision_at_k(relevant, 50, n_relevant),
        })

    rows.append({
        "id": case["id"],
        "category": case["category"],
        "style": case["style"],
        "system": "Random",
        "P@5": prior,
        "P@10": prior,
        "nDCG@10": prior,
        "AP@50": prior,
    })

category_results = pd.DataFrame(rows)

METRICS = ["P@5", "P@10", "nDCG@10", "AP@50"]
SYSTEMS = ["Random", "BM25", "CLIP", "Qwen3-VL", "InternRAG"]

## Results

In [ ]:
def bootstrap_ci(values, n_resamples: int = 10_000, seed: int = 0) -> tuple[float, float]:
    """95% bootstrap confidence interval of the mean over queries."""
    values = np.asarray(values.dropna())
    rng = np.random.default_rng(seed)
    means = rng.choice(values, size=(n_resamples, len(values))).mean(axis=1)
    return np.quantile(means, 0.025), np.quantile(means, 0.975)


summary = {}

for system in SYSTEMS:
    system_results = category_results[category_results["system"] == system]
    summary[system] = {}

    for metric in METRICS:
        values = system_results[metric]
        if values.isna().all():
            summary[system][metric] = "-"
            continue
        low, high = bootstrap_ci(values)
        summary[system][metric] = f"{values.mean():.3f} [{low:.3f}, {high:.3f}]"

pd.DataFrame(summary).T

By query style (`named` vs `descriptive`):

In [ ]:
(
    category_results
    .groupby(["style", "system"])[METRICS]
    .mean()
    .unstack("style")
    .reindex(SYSTEMS)
    .round(3)
)

P@5 by category (the only metric available for all systems):

In [ ]:
(
    category_results
    .pivot_table(index="category", columns="system", values="P@5", aggfunc="mean")
    [SYSTEMS]
    .round(2)
)

## Qualitative Check

Top-5 images of each system for one query. Green titles are relevant, red titles are not (with the actual category).

In [ ]:
def show_top_k(query_id: str, k: int = TOP_K):
    case = next(case for case in CATEGORY_QUERIES if case["id"] == query_id)
    systems = list(rankings[query_id])

    fig, axes = plt.subplots(len(systems), k, figsize=(2.2 * k, 2.4 * len(systems)))
    fig.suptitle(f"{query_id}: {case['query']}  (target: {case['category']})")

    for row, system in enumerate(systems):
        for col in range(k):
            ax = axes[row, col]
            ax.axis("off")

            if col >= len(rankings[query_id][system]):
                continue

            image_id = rankings[query_id][system][col]
            category = CATEGORY_OF.get(image_id)
            ax.imshow(np.load(get_crop_path(image_id)), cmap="gray")
            ax.set_title(
                f"{system}\n{image_id} {category}" if col == 0 else f"\n{image_id} {category}",
                fontsize=7,
                color="green" if category == case["category"] else "red",
            )

    plt.tight_layout()
    plt.show()

show_top_k("NW-03")

Save the rankings and per-query results so the numbers can be reproduced without re-running retrieval.

In [ ]:
CATEGORY_RESULTS_DIR = PROJECT_ROOT / "data" / "evaluation" / "category_retrieval"
CATEGORY_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

category_results.to_csv(CATEGORY_RESULTS_DIR / "results.csv", index=False)

with open(CATEGORY_RESULTS_DIR / "rankings.json", "w", encoding="utf-8") as file:
    json.dump(rankings, file, indent=2)

# 2. Target Retrieval

What you get from it: this is the standard way text-to-image retrieval is evaluated in the literature. COCO and Flickr30k give one caption per image and measure whether the system ranks that image at top-1, top-5 or top-10 (Recall@K, MRR). It measures fine-grained discrimination, which part 1 can’t.

Getting the details right:
- Generate new queries instead. Run Qwen3.6 and Gemma on the image with a different prompt: “write a 1–2 sentence search query a researcher would type to find this image; don’t mention the image ID or metadata”. Short queries are closer to real use and less circular for BM25.
- Sample about 200–300 targets, stratified by category and by your DINO difficulty tiers (mean 5-NN distance). Report Recall@K per tier, since isolated images should be easier than ones with many lookalikes.
- Report a soft hit next to the strict hit. Many crops come from the same sample, so the system may return a near-identical image. Count a hit when the top-K contains the target or an image with DINO cosine similarity above a threshold to it. Report both numbers.

Protocol:
1. Sample target images, stratified by category and by **visual isolation** (how far an image is from its nearest DINO neighbours). Isolated images should be easier to find than images with many lookalikes.
2. For every target, a multimodal LLM looks at the image and writes a short search query, like a researcher would type. The main generator is **Qwen3.6**, not Gemma: Gemma wrote the descriptions indexed by BM25, so Gemma-written queries would share its vocabulary. Gemma queries are also generated as a control to measure how much BM25 gains from that shared vocabulary.
3. Each system retrieves the top `TARGET_DEPTH` images and we look at the rank of the target.

Metrics:
- **R@k** (Recall@k): fraction of queries whose target is in the top k.
- **MRR**: mean of 1/rank of the target (0 if not retrieved within `TARGET_DEPTH`).
- **Strict** counts only the target itself. **Soft** also accepts a near-duplicate of the target (DINO cosine similarity ≥ `NEAR_DUPLICATE_THRESHOLD`): many crops are the same scene slightly shifted, and no text query can distinguish them.

Limitations: a miss does not mean the retrieved images are irrelevant, so this is a lower bound on quality; it measures fine-grained discrimination, while part 1 measures relevance.

## Target Sampling

In [ ]:
TARGETS_PER_CELL = 10  # per (category, isolation tier)
TARGET_DEPTH = 100
NEAR_DUPLICATE_THRESHOLD = 0.90

TARGET_RESULTS_DIR = PROJECT_ROOT / "data" / "evaluation" / "target_retrieval"
TARGET_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
dino_files = sorted(SEMDataset.EMBEDDINGS_DINO.glob("*.npz"))

dino_ids = [file.name.split("_")[0] for file in dino_files]
# Load the cls vector from each file -> DINO’s global [CLS] token, a single vector that summarises the whole image
# vstack stacks the vectors into an (N, D) matrix with one row per image
dino_embeddings = np.vstack([np.load(file)["cls"] for file in dino_files]).astype(np.float32)
# L2-normalises each row to length 1 -> the dot product of two rows equals their cosine similarity
dino_embeddings /= np.linalg.norm(dino_embeddings, axis=1, keepdims=True)

# Lookup table from image ID to row number in the matrix
dino_index = {image_id: i for i, image_id in enumerate(dino_ids)}

print(f"Loaded {len(dino_ids)} DINO embeddings")

Visual isolation = mean cosine distance to the 5 nearest neighbours, split into tertiles (`low` = many lookalikes, `high` = isolated).

In [ ]:
nn = NearestNeighbors(n_neighbors=6, metric="cosine")
nn.fit(dino_embeddings)
distances, _ = nn.kneighbors(dino_embeddings)

# The first neighbour is the image itself
isolation = distances[:, 1:].mean(axis=1)

low_threshold, high_threshold = np.quantile(isolation, [1 / 3, 2 / 3])

image_pool = pd.DataFrame({
    "image_id": dino_ids,
    "category": [CATEGORY_OF[image_id] for image_id in dino_ids],
    "isolation": isolation,
    "tier": np.select(
        [isolation <= low_threshold, isolation <= high_threshold],
        ["low", "medium"],
        default="high",
    ),
})

image_pool.groupby(["category", "tier"]).size().unstack("tier")[["low", "medium", "high"]]

In [ ]:
targets = (
    image_pool
    .sample(frac=1, random_state=0)
    .groupby(["category", "tier"])
    .head(TARGETS_PER_CELL)
    .sort_values(["category", "tier", "image_id"])
    .reset_index(drop=True)
)

print(f"Targets: {len(targets)}")

Near-duplicates of every target (excluding the target itself).

In [ ]:
target_embeddings = dino_embeddings[[dino_index[image_id] for image_id in targets["image_id"]]]
similarities = target_embeddings @ dino_embeddings.T

NEAR_DUPLICATES = {
    image_id: {
        dino_ids[j]
        for j in np.where(similarities[i] >= NEAR_DUPLICATE_THRESHOLD)[0]
        if dino_ids[j] != image_id
    } for i, image_id in enumerate(targets["image_id"])
}

targets["near_duplicates"] = targets["image_id"].map(lambda image_id: len(NEAR_DUPLICATES[image_id]))
targets.groupby("tier")["near_duplicates"].describe()

## Query Generation

Queries are cached in `queries.json`, so they are generated only once and can be reviewed (and fixed by hand) before the final run.

In [ ]:
QUERY_GENERATION_PROMPT = """
Write the search query a materials scientist would type to find exactly this image
in a large database of SEM images.

Describe what is visible and distinctive: the objects, their shape, size relative to
each other, arrangement, texture, contrast and viewing angle.

Rules:
- One or two sentences, at most 40 words.
- Do not mention scale bars, numbers, magnification or instrument settings.
- Do not start with "SEM image of" or similar.
- Return only the query, without quotes.
"""

QUERY_GENERATORS = {
    "qwen3.6": VLLM.QWEN36_MODEL,
    "gemma": VLLM.GEMMA_MODEL,
}

In [ ]:
def generate_target_query(model: str, image_id: str) -> dict:
    response = client.chat.completions.create(
        model=model,
        messages=[
            {
                "role": "user",
                "content": [
                    {"type": "image_url", "image_url": {"url": encode_image(get_crop_path(image_id))}},
                    {"type": "text", "text": QUERY_GENERATION_PROMPT},
                ],
            }
        ],
        temperature=0.0,
        max_tokens=150,
        extra_body={"chat_template_kwargs": {"enable_thinking": False}},
    )

    choice = response.choices[0]

    return {
        "query": choice.message.content.strip().strip('"'),
        "finish_reason": choice.finish_reason,
    }

In [ ]:
queries_path = TARGET_RESULTS_DIR / "queries.json"

if queries_path.exists():
    with open(queries_path, "r", encoding="utf-8") as file:
        target_queries = json.load(file)
else:
    target_queries = {}

for generator, model in QUERY_GENERATORS.items():
    generated = target_queries.setdefault(generator, {})

    for image_id in targets["image_id"]:
        if image_id not in generated:
            generated[image_id] = generate_target_query(model, image_id)

    with open(queries_path, "w", encoding="utf-8") as file:
        json.dump(target_queries, file, indent=2, ensure_ascii=False)

for generator, generated in target_queries.items():
    truncated = sum(query["finish_reason"] != "stop" for query in generated.values())
    print(f"{generator}: {len(generated)} queries, {truncated} truncated")

In [ ]:
for image_id in targets["image_id"].sample(5, random_state=1):
    print(f"{image_id} ({CATEGORY_OF[image_id]})")
    for generator in QUERY_GENERATORS:
        print(f"  {generator:8s}: {target_queries[generator][image_id]['query']}")

## Retrieval

In [ ]:
def retrieve_all_deep(query: str, depth: int = TARGET_DEPTH) -> dict[str, list[str]]:
    """Return the ranked image IDs of every system for a text query."""
    bm25 = [result["image_id"] for result in bm25_retriever.search(query, top_k=depth)]

    clip_embedding = clip_embedder.embed_text(query)
    clip = [doc.id for doc in clip_retriever.run(query_embedding=clip_embedding, top_k=depth)["documents"]]

    qwen_embedding = qwen_embedder.run(text=query)["embedding"]
    qwen_docs = qwen_retriever.run(query_embedding=qwen_embedding, top_k=depth)["documents"]
    internrag_docs = mmr.run(query_embedding=qwen_embedding, documents=qwen_docs[:EVAL_DEPTH])["documents"]

    return {
        "BM25": bm25,
        "CLIP": clip,
        "Qwen3-VL": [doc.id for doc in qwen_docs],
        "InternRAG": [doc.id for doc in internrag_docs],
    }

In [ ]:
target_rankings = {
    generator: {
        image_id: retrieve_all_deep(target_queries[generator][image_id]["query"])
        for image_id in targets["image_id"]
    } for generator in QUERY_GENERATORS
}

## Metrics

In [ ]:
def first_hit_rank(ranking: list[str], hits: set[str]) -> int | None:
    """1-based rank of the first image in `hits`, or None if none was retrieved."""
    for rank, image_id in enumerate(ranking, start=1):
        if image_id in hits:
            return rank
    return None


def recall_at_k(rank: int | None, k: int, depth: int) -> float:
    if depth < k:
        return np.nan
    return float(rank is not None and rank <= k)

In [ ]:
TARGET_KS = [1, 5, 10, 50]

rows = []

for generator in QUERY_GENERATORS:
    for target in targets.itertuples():
        image_id = target.image_id

        for system, ranking in target_rankings[generator][image_id].items():
            for match, hits in [
                ("strict", {image_id}),
                ("soft", {image_id} | NEAR_DUPLICATES[image_id]),
            ]:
                rank = first_hit_rank(ranking, hits)

                row = {
                    "generator": generator,
                    "image_id": image_id,
                    "category": target.category,
                    "tier": target.tier,
                    "system": system,
                    "match": match,
                    "rank": rank,
                    "MRR": np.nan if system == "InternRAG" else (1 / rank if rank else 0.0),
                }
                for k in TARGET_KS:
                    row[f"R@{k}"] = recall_at_k(rank, k, len(ranking))

                rows.append(row)

target_results = pd.DataFrame(rows)

TARGET_METRICS = [f"R@{k}" for k in TARGET_KS] + ["MRR"]
TARGET_SYSTEMS = ["BM25", "CLIP", "Qwen3-VL", "InternRAG"]

## Results

Main result: Qwen3.6 queries, with 95% bootstrap confidence intervals.

In [ ]:
def summarize(results: pd.DataFrame) -> pd.DataFrame:
    summary = {}

    for system in TARGET_SYSTEMS:
        system_results = results[results["system"] == system]
        summary[system] = {}

        for metric in TARGET_METRICS:
            values = system_results[metric]
            if values.isna().all():
                summary[system][metric] = "-"
                continue
            low, high = bootstrap_ci(values)
            summary[system][metric] = f"{values.mean():.3f} [{low:.3f}, {high:.3f}]"

    return pd.DataFrame(summary).T


main_results = target_results[target_results["generator"] == "qwen3.6"]

for match in ["strict", "soft"]:
    print(f"{match.upper()} match")
    display(summarize(main_results[main_results["match"] == match]))

Effect of the query generator. If BM25 gains much more than the other systems with Gemma queries, it benefits from sharing Gemma's vocabulary.

In [ ]:
(
    target_results[target_results["match"] == "strict"]
    .groupby(["system", "generator"])[["R@1", "R@10", "MRR"]]
    .mean()
    .unstack("generator")
    .reindex(TARGET_SYSTEMS)
    .round(3)
)

R@10 by visual isolation tier (Qwen3.6 queries). Soft match is easier in the `low` tier by construction, since those targets have many near-duplicates, so compare tiers on the strict match.

In [ ]:
(
    main_results
    .pivot_table(index=["match", "tier"], columns="system", values="R@10", aggfunc="mean")
    .reindex(pd.MultiIndex.from_product([["strict", "soft"], ["low", "medium", "high"]]))
    [TARGET_SYSTEMS[:-1]]
    .round(3)
)

By category (Qwen3.6 queries, soft match, R@10):

In [ ]:
(
    main_results[main_results["match"] == "soft"]
    .pivot_table(index="category", columns="system", values="R@10", aggfunc="mean")
    [TARGET_SYSTEMS[:-1]]
    .round(2)
)

## Qualitative Check

Target image, its query, and the top-5 images of each system. Green = target, orange = near-duplicate.

In [ ]:
def show_target(image_id: str, generator: str = "qwen3.6", k: int = TOP_K):
    systems = list(target_rankings[generator][image_id])
    query = target_queries[generator][image_id]["query"]

    fig, axes = plt.subplots(len(systems) + 1, k, figsize=(2.2 * k, 2.4 * (len(systems) + 1)))
    fig.suptitle(f"{generator}: {query}", fontsize=9, wrap=True)

    for ax in axes.flat:
        ax.axis("off")

    axes[0, 0].imshow(np.load(get_crop_path(image_id)), cmap="gray")
    axes[0, 0].set_title(f"TARGET {image_id}", fontsize=7)

    for row, system in enumerate(systems, start=1):
        for col, retrieved_id in enumerate(target_rankings[generator][image_id][system][:k]):
            if retrieved_id == image_id:
                color = "green"
            elif retrieved_id in NEAR_DUPLICATES[image_id]:
                color = "orange"
            else:
                color = "black"

            ax = axes[row, col]
            ax.imshow(np.load(get_crop_path(retrieved_id)), cmap="gray")
            ax.set_title(f"{system}\n{retrieved_id}" if col == 0 else f"\n{retrieved_id}", fontsize=7, color=color)

    plt.tight_layout()
    plt.show()

show_target(targets["image_id"].iloc[40])

In [ ]:
target_results.to_csv(TARGET_RESULTS_DIR / "results.csv", index=False)

with open(TARGET_RESULTS_DIR / "rankings.json", "w", encoding="utf-8") as file:
    json.dump(target_rankings, file, indent=2)